In [ ]:
import os
print("Working folder:", os.getcwd())
for d in [".", "/content", "/content/drive/MyDrive", "/kaggle/input"]:
    if os.path.isdir(d):
        print("\n", d, "->", os.listdir(d)[:30])

Working folder: /content

 . -> ['.config', 'sample_data']

 /content -> ['.config', 'sample_data']

 /kaggle/input -> []


In [ ]:
import os
print("Working folder:", os.getcwd())
for d in [".", "/content", "/content/drive/MyDrive", "/kaggle/input"]:
    if os.path.isdir(d):
        print("\n", d, "->", os.listdir(d)[:30])

Working folder: /content

 . -> ['.config', 'sample_data']

 /content -> ['.config', 'sample_data']

 /kaggle/input -> []


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
root = "/content/drive/MyDrive"
for folder, subs, files in os.walk(root):
    hits = [f for f in files if f.startswith("yellow_tripdata") and f.endswith(".parquet")]
    if hits:
        print(folder, "->", len(hits), "files, e.g.", sorted(hits)[0])
    if folder.count("/") - root.count("/") >= 4:   # don't dig deeper than 4 levels
        subs.clear()

/content/drive/MyDrive/BigData Project/out_2023 -> 1 files, e.g. yellow_tripdata_2023-12.parquet


In [ ]:
import os, duckdb

root = "/content/drive/MyDrive"
found = {}
for folder, subs, files in os.walk(root):
    n = [f for f in files if f.startswith("yellow_tripdata") and f.endswith(".parquet")]
    if n:
        found[folder] = len(n)
    if folder.count("/") - root.count("/") >= 4:
        subs.clear()

print("Folders with yellow_tripdata files:", found)

if found:
    folder = max(found, key=found.get)          # the folder with the most files
    print("Using:", folder)
    duckdb.sql(f"""
        SELECT count(*) AS n_rows,
               min(tpep_pickup_datetime) AS first_trip,
               max(tpep_pickup_datetime) AS last_trip
        FROM read_parquet('{folder}/yellow_tripdata_*.parquet', union_by_name=true)
    """).show()
else:
    print("No files found. Top level of your Drive:", os.listdir(root)[:40])

Folders with yellow_tripdata files: {'/content/drive/MyDrive/BigData Project/out_2023': 1}
Using: /content/drive/MyDrive/BigData Project/out_2023
┌─────────┬─────────────────────┬─────────────────────┐
│ n_rows  │     first_trip      │      last_trip      │
│  int64  │      timestamp      │      timestamp      │
├─────────┼─────────────────────┼─────────────────────┤
│ 3376567 │ 2002-12-31 22:16:54 │ 2024-01-03 19:42:57 │
└─────────┴─────────────────────┴─────────────────────┘



In [ ]:
import os, re
root = "/content/drive/MyDrive"
for folder, subs, files in os.walk(root):
    n = sorted(f for f in files if f.startswith("yellow_tripdata") and f.endswith(".parquet"))
    if n:
        print(folder, "->", len(n), "files:", n[0], "...", n[-1])
    if folder.count("/") - root.count("/") >= 4:
        subs.clear()

/content/drive/MyDrive/BigData Project/out_2023 -> 1 files: yellow_tripdata_2023-12.parquet ... yellow_tripdata_2023-12.parquet


In [ ]:
import duckdb
base = "/content/drive/MyDrive/BigData Project"
duckdb.sql(f"""
    SELECT count(*) AS n_rows,
           min(tpep_pickup_datetime) AS first_trip,
           max(tpep_pickup_datetime) AS last_trip
    FROM read_parquet('{base}/*/yellow_tripdata_*.parquet', union_by_name=true)
""").show()

┌─────────┬─────────────────────┬─────────────────────┐
│ n_rows  │     first_trip      │      last_trip      │
│  int64  │      timestamp      │      timestamp      │
├─────────┼─────────────────────┼─────────────────────┤
│ 3376567 │ 2002-12-31 22:16:54 │ 2024-01-03 19:42:57 │
└─────────┴─────────────────────┴─────────────────────┘



In [ ]:
%%writefile nyc_taxi_clean.py
#!/usr/bin/env python3
"""NYC Yellow Taxi - cleaning + enrichment + small aggregates (PySpark)."""
import argparse
import glob
import os
import re
import sys

from pyspark.sql import SparkSession, functions as F, types as T

CANON = {
    "vendorid": "int", "tpep_pickup_datetime": "timestamp", "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "int", "trip_distance": "double", "ratecodeid": "int",
    "store_and_fwd_flag": "string", "pulocationid": "int", "dolocationid": "int",
    "payment_type": "int", "fare_amount": "double", "extra": "double", "mta_tax": "double",
    "tip_amount": "double", "tolls_amount": "double", "improvement_surcharge": "double",
    "total_amount": "double", "congestion_surcharge": "double", "airport_fee": "double",
    "cbd_congestion_fee": "double",
}
RATECODE = {1: "Standard", 2: "JFK", 3: "Newark", 4: "Nassau/Westchester", 5: "Negotiated fare", 6: "Group ride"}
PAYMENT = {0: "Flex fare", 1: "Credit card", 2: "Cash", 3: "No charge", 4: "Dispute", 5: "Unknown", 6: "Voided trip"}


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--input", required=True)
    p.add_argument("--zones", required=True)
    p.add_argument("--weather")
    p.add_argument("--out", required=True)
    p.add_argument("--only")
    p.add_argument("--master", default="local[*]")
    p.add_argument("--driver-memory", default="8g")
    p.add_argument("--shuffle-partitions", type=int, default=200)
    p.add_argument("--start-date", default="2022-01-01")
    p.add_argument("--end-date", default="2026-01-01")
    p.add_argument("--max-hours", type=float, default=24.0)
    p.add_argument("--max-miles", type=float, default=100.0)
    p.add_argument("--max-fare", type=float, default=500.0)
    p.add_argument("--max-mph", type=float, default=80.0)
    p.add_argument("--no-month-check", action="store_true")
    p.add_argument("--dedupe", action="store_true")
    return p.parse_args()


def load_month_file(spark, path):
    m = re.search(r"(\d{4})-(\d{2})", os.path.basename(path))
    if not m:
        raise ValueError("cannot read year-month from file name: " + path)
    fy, fm = int(m.group(1)), int(m.group(2))
    df = spark.read.parquet(path)
    df = df.toDF(*[c.lower() for c in df.columns])
    cols = []
    for name, typ in CANON.items():
        if name in df.columns:
            cols.append(F.col(name).cast(typ).alias(name))
        else:
            cols.append(F.lit(None).cast(typ).alias(name))
    return df.select(*cols).withColumn("file_year", F.lit(fy)).withColumn("file_month", F.lit(fm))


def main():
    a = parse_args()
    spark = (SparkSession.builder.master(a.master).appName("nyc_taxi_clean")
             .config("spark.driver.memory", a.driver_memory)
             .config("spark.sql.shuffle.partitions", str(a.shuffle_partitions))
             .config("spark.sql.session.timeZone", "UTC")
             .getOrCreate())
    spark.sparkContext.setLogLevel("WARN")

    files = sorted(glob.glob(os.path.join(a.input, "**", "yellow_tripdata_*.parquet"), recursive=True))
    if a.only:
        keep = set(a.only.split(","))
        files = [f for f in files if re.search(r"(\d{4}-\d{2})", os.path.basename(f)).group(1) in keep]
    if not files:
        sys.exit("No yellow_tripdata_*.parquet files found (check --input / --only)")
    print("Files:", len(files))

    df = None
    for f in files:
        d = load_month_file(spark, f)
        df = d if df is None else df.unionByName(d)
    if a.dedupe:
        df = df.dropDuplicates()

    z = (spark.read.option("header", True).csv(a.zones)
         .select(F.col("LocationID").cast("int").alias("zid"),
                 F.col("Borough").alias("borough"), F.col("Zone").alias("zone")))
    zpu = F.broadcast(z.select(F.col("zid").alias("pu_zid"), F.col("borough").alias("pu_borough"), F.col("zone").alias("pu_zone")))
    zdo = F.broadcast(z.select(F.col("zid").alias("do_zid"), F.col("borough").alias("do_borough"), F.col("zone").alias("do_zone")))
    df = (df.join(zpu, F.col("pulocationid") == F.col("pu_zid"), "left")
            .join(zdo, F.col("dolocationid") == F.col("do_zid"), "left"))

    pu, do = F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")
    df = (df.withColumn("trip_duration_min", (F.unix_timestamp(do) - F.unix_timestamp(pu)) / 60.0)
            .withColumn("speed_mph", F.when((F.col("trip_duration_min") > 0) & (F.col("trip_distance") > 0),
                                            F.col("trip_distance") / (F.col("trip_duration_min") / 60.0))))

    money = ["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
             "improvement_surcharge", "total_amount", "congestion_surcharge", "airport_fee", "cbd_congestion_fee"]
    neg_any = None
    for c in money:
        e = F.col(c) < 0
        neg_any = e if neg_any is None else (neg_any | e)

    rules = [
        ("r01_missing_key_fields",
         pu.isNull() | do.isNull() | F.col("pulocationid").isNull() | F.col("dolocationid").isNull()
         | F.col("trip_distance").isNull() | F.col("fare_amount").isNull() | F.col("total_amount").isNull()),
        ("r02_location_id_not_in_zone_lookup", F.col("pu_zid").isNull() | F.col("do_zid").isNull()),
        ("r03_malformed_timestamp_outside_study_window",
         (pu < F.lit(a.start_date)) | (pu >= F.lit(a.end_date)) | (do < F.lit(a.start_date))
         | (do >= F.date_add(F.lit(a.end_date).cast("date"), 1))),
    ]
    if not a.no_month_check:
        rules.append(("r04_pickup_not_in_file_month",
                      (F.year(pu) != F.col("file_year")) | (F.month(pu) != F.col("file_month"))))
    rules += [
        ("r05_dropoff_not_after_pickup", F.col("trip_duration_min") <= 0),
        ("r06_duration_over_max", F.col("trip_duration_min") > a.max_hours * 60),
        ("r07_zero_or_negative_distance", F.col("trip_distance") <= 0),
        ("r08_distance_over_max", F.col("trip_distance") > a.max_miles),
        ("r09_fare_zero_or_negative", F.col("fare_amount") <= 0),
        ("r10_fare_over_max", F.col("fare_amount") > a.max_fare),
        ("r11_negative_charge_or_total", neg_any | (F.col("total_amount") <= 0)),
        ("r12_passenger_count_out_of_range", F.col("passenger_count").isNotNull()
         & ((F.col("passenger_count") < 1) | (F.col("passenger_count") > 6))),
        ("r13_implausible_speed", F.col("speed_mph") > a.max_mph),
    ]
    rules = [(n, F.coalesce(e, F.lit(False))) for n, e in rules]
    names = [n for n, _ in rules]

    first_fail = F.coalesce(*[F.when(e, F.lit(n)) for n, e in rules])
    df = df.withColumn("_drop_reason", first_fail)

    aggs = [F.count(F.lit(1)).alias("rows_in"),
            F.sum(F.when(F.col("_drop_reason").isNull(), 1).otherwise(0)).alias("rows_kept")]
    for n, e in rules:
        aggs.append(F.sum(F.when(e, 1).otherwise(0)).alias("fails__" + n))
        aggs.append(F.sum(F.when(F.col("_drop_reason") == n, 1).otherwise(0)).alias("dropped_first__" + n))
    log = df.groupBy("file_year", "file_month").agg(*aggs).orderBy("file_year", "file_month")
    log.cache()
    log.coalesce(1).write.mode("overwrite").option("header", True).csv(os.path.join(a.out, "cleaning_log_by_month_csv"))
    tot = log.agg(*[F.sum(c).alias(c) for c in log.columns if c not in ("file_year", "file_month")]).collect()[0].asDict()
    rows_in, rows_kept = tot["rows_in"], tot["rows_kept"]
    summ = [("TOTAL rows in", rows_in, None, None), ("TOTAL rows kept", rows_kept, None, None)]
    for n in names:
        summ.append((n, tot["fails__" + n], tot["dropped_first__" + n],
                     round(100.0 * tot["dropped_first__" + n] / max(rows_in, 1), 4)))
    summ_schema = T.StructType([T.StructField("item", T.StringType()),
                                T.StructField("rows_failing_rule_any", T.LongType()),
                                T.StructField("rows_dropped_first_rule", T.LongType()),
                                T.StructField("pct_of_rows_in", T.DoubleType())])
    summ_df = spark.createDataFrame(summ, summ_schema)
    summ_df.coalesce(1).write.mode("overwrite").option("header", True).csv(os.path.join(a.out, "cleaning_log_summary_csv"))
    print("\n=== CLEANING SUMMARY ===")
    for r in summ:
        print(r)

    c = df.filter(F.col("_drop_reason").isNull()).drop("_drop_reason", "pu_zid", "do_zid")
    c = (c.withColumn("passenger_missing", F.col("passenger_count").isNull())
          .withColumn("fee_fields_missing", F.col("congestion_surcharge").isNull() | F.col("airport_fee").isNull())
          .withColumn("cbd_congestion_fee", F.coalesce(F.col("cbd_congestion_fee"), F.lit(0.0)))
          .withColumn("congestion_surcharge", F.coalesce(F.col("congestion_surcharge"), F.lit(0.0)))
          .withColumn("airport_fee", F.coalesce(F.col("airport_fee"), F.lit(0.0)))
          .withColumn("ratecodeid", F.when(F.col("ratecodeid").between(1, 6), F.col("ratecodeid")))
          .withColumn("payment_type", F.when(F.col("payment_type").between(0, 6), F.col("payment_type"))))
    rc = F.create_map(*[x for k, v in RATECODE.items() for x in (F.lit(k), F.lit(v))])
    pm = F.create_map(*[x for k, v in PAYMENT.items() for x in (F.lit(k), F.lit(v))])
    c = (c.withColumn("ratecode_desc", F.coalesce(rc[F.col("ratecodeid")], F.lit("Unknown")))
          .withColumn("payment_desc", F.coalesce(pm[F.col("payment_type")], F.lit("Unknown")))
          .withColumn("pickup_date", F.to_date(F.col("tpep_pickup_datetime")))
          .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
          .withColumn("pickup_dow", F.dayofweek("tpep_pickup_datetime"))
          .withColumn("year", F.year("tpep_pickup_datetime"))
          .withColumn("month", F.month("tpep_pickup_datetime"))
          .withColumn("is_weekend", F.dayofweek("tpep_pickup_datetime").isin(1, 7))
          .withColumn("has_congestion_surcharge", F.col("congestion_surcharge") > 0)
          .withColumn("has_cbd_fee", F.col("cbd_congestion_fee") > 0)
          .withColumn("post_congestion_pricing", F.col("pickup_date") >= F.lit("2025-01-05").cast("date"))
          .withColumn("tip_pct_card", F.when((F.col("payment_type") == 1) & (F.col("fare_amount") > 0),
                                             100.0 * F.col("tip_amount") / F.col("fare_amount"))))

    wcols = []
    if a.weather:
        w = spark.read.option("header", True).option("inferSchema", True).csv(a.weather)
        w = w.toDF(*[x.strip().upper() for x in w.columns])
        wcols = [x for x in ("PRCP", "SNOW", "SNWD", "TMAX", "TMIN", "TAVG", "AWND") if x in w.columns]
        w = (w.withColumn("wdate", F.to_date(F.col("DATE")))
               .groupBy("wdate").agg(*[F.avg(F.col(x).cast("double")).alias("w_" + x.lower()) for x in wcols]))
        c = c.join(F.broadcast(w), c.pickup_date == w.wdate, "left").drop("wdate")
        if "PRCP" in wcols:
            c = c.withColumn("rain_day", F.col("w_prcp") > 0)
        if "SNOW" in wcols:
            c = c.withColumn("snow_day", F.col("w_snow") > 0)

    cleaned_path = os.path.join(a.out, "cleaned_trips")
    c.write.mode("overwrite").partitionBy("year", "month").parquet(cleaned_path)
    print("Wrote cleaned trips ->", cleaned_path)

    cc = spark.read.parquet(cleaned_path)
    wsel = ["w_" + x.lower() for x in wcols] + (["rain_day"] if "PRCP" in wcols else []) + (["snow_day"] if "SNOW" in wcols else [])
    metrics = [F.count(F.lit(1)).alias("trips"),
               F.avg("fare_amount").alias("avg_fare"),
               F.avg("tip_pct_card").alias("avg_tip_pct_card"),
               F.avg("trip_distance").alias("avg_distance_mi"),
               F.avg("trip_duration_min").alias("avg_duration_min"),
               F.sum("fare_amount").alias("sum_fare"),
               F.sum("total_amount").alias("sum_total_amount"),
               F.avg(F.col("has_congestion_surcharge").cast("double")).alias("share_with_congestion_surcharge"),
               F.avg(F.col("has_cbd_fee").cast("double")).alias("share_with_cbd_fee"),
               F.sum("cbd_congestion_fee").alias("sum_cbd_fee")]

    def with_weather(g):
        if not wcols:
            return g
        wd = cc.select("pickup_date", *wsel).dropDuplicates(["pickup_date"])
        return g.join(wd, "pickup_date", "left")

    hz = with_weather(cc.groupBy("pickup_date", "pickup_hour", "pulocationid", "pu_borough", "pu_zone").agg(*metrics))
    hz.write.mode("overwrite").parquet(os.path.join(a.out, "agg_hourly_zone"))
    db = with_weather(cc.groupBy("pickup_date", "pu_borough").agg(*metrics)).orderBy("pickup_date", "pu_borough")
    db.coalesce(1).write.mode("overwrite").option("header", True).csv(os.path.join(a.out, "agg_daily_borough_csv"))
    print("Done. Share the small CSV folders: cleaning_log_*_csv and agg_daily_borough_csv.")
    spark.stop()


if __name__ == "__main__":
    main()



Writing nyc_taxi_clean.py


In [ ]:
import os, urllib.request
dest = "/content/drive/MyDrive/BigData Project/taxi_zone_lookup.csv"
url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
urllib.request.urlretrieve(url, dest)
print("exists:", os.path.exists(dest), "| size:", os.path.getsize(dest), "bytes")
print(open(dest).read()[:200])

exists: True | size: 12331 bytes
"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"
3,"Bronx","Allerton/Pelham Gardens","Boro Zone"
4,"Manhattan","Alphabet City","Yellow Z


In [ ]:
!pip install -q pyspark
!python nyc_taxi_clean.py \
    --input "/content/drive/MyDrive/BigData Project" \
    --zones "/content/drive/MyDrive/BigData Project/taxi_zone_lookup.csv" \
    --out "/content/drive/MyDrive/BigData Project/out_test" \
    --only 2025-01 --driver-memory 6g

[0.029s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.029s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 04:53:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where

In [ ]:
import subprocess
base = "/content/drive/MyDrive/BigData Project"
for y in [2022, 2023, 2024, 2025]:
    months = ",".join(f"{y}-{m:02d}" for m in range(1, 13))
    subprocess.run(
        f'python nyc_taxi_clean.py --input "{base}" --zones "{base}/taxi_zone_lookup.csv" '
        f'--out "{base}/out_{y}" --only {months} --driver-memory 6g 2>&1 | grep -v "^\\s*at \\|cgroup"',
        shell=True)
    print("finished", y)

finished 2022
finished 2023
finished 2024
finished 2025


In [ ]:
s = open("nyc_taxi_clean.py").read()
a = '("r09_fare_zero_or_negative", F.col("fare_amount") <= 0),'
b = '("r11_negative_charge_or_total", neg_any | (F.col("total_amount") <= 0)),'
c = '.withColumn("has_cbd_fee"'

# Safely patch only if the unpatched patterns exist, otherwise assume it is already patched
patched = False
if a in s:
    s = s.replace(a, '("r09_fare_negative", F.col("fare_amount") < 0),')
    patched = True
if b in s:
    s = s.replace(b, '("r11_negative_charge_or_total", neg_any | (F.col("total_amount") < 0)),')
    patched = True
if c in s and '.withColumn("is_zero_fare"' not in s:
    s = s.replace(c, '.withColumn("is_zero_fare", F.col("fare_amount") == 0)\n          ' + c, 1)
    patched = True

if patched:
    open("nyc_taxi_clean.py", "w").write(s)
    print("patched")
else:
    print("Already patched or target text not found")

patched


In [ ]:
import pandas as pd
url = "https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/USW00094728.csv"
dest = "/content/drive/MyDrive/BigData Project/noaa_nyc_daily.csv"
w = pd.read_csv(url, low_memory=False)
w = w[(w["DATE"] >= "2022-01-01") & (w["DATE"] <= "2025-12-31")]
w.to_csv(dest, index=False)
print(len(w), "rows;", w["DATE"].min(), "to", w["DATE"].max())
print(w[["DATE", "PRCP", "SNOW", "TMAX", "TMIN"]].head(5))

1461 rows; 2022-01-01 to 2025-12-31
             DATE  PRCP  SNOW   TMAX   TMIN
55882  2022-01-01   201   0.0  133.0  100.0
55883  2022-01-02    10   0.0  150.0   28.0
55884  2022-01-03     0   0.0   28.0  -55.0
55885  2022-01-04     0   0.0   11.0  -71.0
55886  2022-01-05    58   0.0   83.0   -5.0


In [ ]:
import duckdb, pandas as pd
base = "/content/drive/MyDrive/BigData Project"; fin = f"{base}/FINAL_for_partner"
con = duckdb.connect()
print(con.sql(f"SELECT count(*) AS n_days, min(pickup_date) AS first_day, max(pickup_date) AS last_day FROM read_csv_auto('{fin}/agg_daily_borough.csv')").df())
print(con.sql(f"SELECT count(*) AS n_rows FROM read_parquet('{fin}/agg_hourly_zone.parquet')").df())

D = [
("cleaned_trips (out_YYYY/cleaned_trips)", "tpep_pickup_datetime / tpep_dropoff_datetime", "Trip start / end, NYC local clock time"),
("cleaned_trips", "pickup_date, pickup_hour, pickup_dow, is_weekend", "Derived from pickup time. pickup_dow: 1=Sunday ... 7=Saturday"),
("cleaned_trips", "pulocationid / dolocationid", "TLC zone IDs"),
("cleaned_trips", "pu_borough, pu_zone, do_borough, do_zone", "Names from the TLC zone lookup. IDs 264/265 = Unknown/N/A, kept"),
("cleaned_trips", "trip_distance", "Miles"),
("cleaned_trips", "trip_duration_min, speed_mph", "Derived from timestamps and distance"),
("cleaned_trips", "passenger_count", "May be blank (not recorded). Counts 0 or above 6 were removed"),
("cleaned_trips", "passenger_missing", "True if passenger_count was blank"),
("cleaned_trips", "fare_amount, tip_amount, tolls_amount, total_amount, extra, mta_tax, improvement_surcharge", "US dollars. Negative amounts removed"),
("cleaned_trips", "congestion_surcharge", "Older NYS surcharge (all years). Blank filled with 0"),
("cleaned_trips", "airport_fee", "Blank filled with 0"),
("cleaned_trips", "fee_fields_missing", "True if congestion_surcharge or airport_fee was originally blank"),
("cleaned_trips", "cbd_congestion_fee", "Manhattan congestion toll, started 5 Jan 2025. 0 before that"),
("cleaned_trips", "has_congestion_surcharge, has_cbd_fee", "True if the charge is above 0"),
("cleaned_trips", "post_congestion_pricing", "True for trips on or after 5 Jan 2025"),
("cleaned_trips", "is_zero_fare", "True if fare_amount = 0. Kept, not removed. Filter these for fare analysis"),
("cleaned_trips", "ratecodeid, ratecode_desc", "Rate code, invalid codes (e.g. 99) set blank and labelled Unknown"),
("cleaned_trips", "payment_type, payment_desc", "Payment type, invalid codes set blank and labelled Unknown"),
("cleaned_trips", "tip_pct_card", "Tip as % of fare, card payments only. Cash tips are not recorded in TLC data"),
("cleaned_trips", "file_year, file_month, year, month", "Source file month and pickup year/month"),
("cleaned_trips", "weather", "NOT included. Join weather_daily.csv on pickup_date = date"),
("agg_hourly_zone / agg_daily_borough", "trips", "Number of cleaned trips in the group"),
("agg_hourly_zone / agg_daily_borough", "avg_fare, sum_fare, sum_total_amount", "Include zero-fare trips"),
("agg_hourly_zone / agg_daily_borough", "avg_tip_pct_card, avg_distance_mi, avg_duration_min", "Group averages"),
("agg_hourly_zone / agg_daily_borough", "share_with_congestion_surcharge, share_with_cbd_fee, sum_cbd_fee", "Congestion charge measures"),
("agg_hourly_zone / agg_daily_borough", "prcp_mm, snow_mm, snow_depth_mm, tmax_c, tmin_c, wind_ms, rain_day, snow_day", "NOAA Central Park daily weather, metric. Joined by date"),
]
pd.DataFrame(D, columns=["file", "column(s)", "meaning"]).to_csv(f"{fin}/DATA_DICTIONARY.csv", index=False)

R = [
("r03_malformed_timestamp_outside_study_window", "Pickup/dropoff outside 2022-2025 (e.g. year 2001)", 670),
("r04_pickup_not_in_file_month", "Pickup month differs from the file's month", 2162),
("r05_dropoff_not_after_pickup", "Duration zero or negative", 607413),
("r06_duration_over_max", "Trip longer than 24 hours", 998),
("r07_zero_or_negative_distance", "Distance 0 or below", 3470766),
("r08_distance_over_max", "Distance over 100 miles", 7110),
("r09_fare_negative", "Fare below 0 (zero fares are kept)", 3836525),
("r10_fare_over_max", "Fare over 500 dollars", 880),
("r11_negative_charge_or_total", "Any other negative charge or negative total", 11036),
("r12_passenger_count_out_of_range", "Passenger count 0 or above 6 (blank is kept)", 1948290),
("r13_implausible_speed", "Speed over 80 mph", 81771),
]
pd.DataFrame(R, columns=["rule", "what_is_removed", "rows_removed"]).to_csv(f"{fin}/CLEANING_RULES.csv", index=False)
print("written")

IOException: IO Error: No files found that match the pattern "/content/drive/MyDrive/BigData Project/FINAL_for_partner/agg_daily_borough.csv"

In [ ]:
import duckdb
fin = "/content/drive/MyDrive/BigData Project/FINAL_for_partner"
f = f"read_csv_auto('{fin}/agg_daily_borough.csv')"
print(duckdb.sql(f"SELECT count(DISTINCT pickup_date) AS distinct_days FROM {f}").df())
print(duckdb.sql(f"""
    SELECT pu_borough,
           pickup_date >= DATE '2025-01-05' AS after_toll,
           round(avg(share_with_cbd_fee), 3) AS share_trips_charged,
           sum(trips) AS trips
    FROM {f}
    WHERE pu_borough IN ('Manhattan', 'Brooklyn', 'Queens')
    GROUP BY 1, 2 ORDER BY 1, 2""").df())

IOException: IO Error: No files found that match the pattern "/content/drive/MyDrive/BigData Project/FINAL_for_partner/agg_daily_borough.csv"

In [ ]:
import duckdb, os, pandas as pd
base = "/content/drive/MyDrive/BigData Project"
fin = f"{base}/FINAL_for_partner"
con = duckdb.connect()

print("FILES IN FINAL_for_partner:")
for f in sorted(os.listdir(fin)):
    print(f"  {f:40s} {os.path.getsize(os.path.join(fin,f))/1e6:9.2f} MB")
print("\nout_test still exists:", os.path.isdir(f"{base}/out_test"))

# weather units
w = pd.read_csv(f"{fin}/weather_daily.csv")
dcol = [c for c in w.columns if "date" in c.lower()][0]
print("\nweather_daily columns:", list(w.columns))
print(w[["prcp_mm","snow_mm","snow_depth_mm","tmax_c","tmin_c","wind_ms"]].agg(["min","max"]).round(1))
print(w.nlargest(5, "snow_mm")[[dcol, "snow_mm", "snow_depth_mm"]])

# columns in the two aggregates
for n, src in [("agg_hourly_zone", f"read_parquet('{fin}/agg_hourly_zone.parquet')"),
               ("agg_daily_borough", f"read_csv_auto('{fin}/agg_daily_borough.csv')")]:
    print(f"\n{n} columns:", list(con.sql(f"DESCRIBE SELECT * FROM {src}").df()["column_name"]))

# full trip count and zero-fare flag (takes a few minutes on Drive)
trips = f"read_parquet('{base}/out_20*/cleaned_trips/**/*.parquet', hive_partitioning=true, union_by_name=true)"
print(con.sql(f"""SELECT count(*) AS trips,
                         sum(CASE WHEN is_zero_fare THEN 1 ELSE 0 END) AS zero_fare_trips
                  FROM {trips}""").df())

FILES IN FINAL_for_partner:
  CLEANING_RULES.csv                            0.00 MB
  DATA_DICTIONARY.csv                           0.00 MB
  agg_daily_borough.csv                         2.13 MB
  agg_hourly_zone.parquet                     171.50 MB
  cleaning_log_by_month.csv                     0.01 MB
  cleaning_log_summary_all_years.csv            0.00 MB
  weather_daily.csv                             0.07 MB

out_test still exists: True

weather_daily columns: ['date', 'prcp_mm', 'snow_mm', 'snow_depth_mm', 'tmax_c', 'tmin_c', 'wind_ms', 'rain_day', 'snow_day']
     prcp_mm  snow_mm  snow_depth_mm  tmax_c  tmin_c  wind_ms
min      0.0      0.0            0.0    -9.3   -16.0      0.3
max    139.2    185.0          180.0    37.2    27.2      6.4
            date  snow_mm  snow_depth_mm
28    2022-01-29    185.0          150.0
6     2022-01-07    147.0          150.0
773   2024-02-13     81.0           30.0
1134  2025-02-08     76.0            0.0
1443  2025-12-14     74.0        

In [ ]:
import shutil, pandas as pd
base = "/content/drive/MyDrive/BigData Project"
fin = f"{base}/FINAL_for_partner"

# 1. Remove the old test output (built with the OLD r09 rule)
shutil.rmtree(f"{base}/out_test")
print("removed out_test")

# 2. Add r01 and r02 (0 rows removed) to CLEANING_RULES.csv
r = pd.read_csv(f"{fin}/CLEANING_RULES.csv")
if not r["rule"].str.startswith("r01").any():
    extra = pd.DataFrame([
        ("r01_missing_key_fields", "Missing pickup/dropoff time, zone IDs, distance, fare or total", 0),
        ("r02_location_id_not_in_zone_lookup", "Zone ID not found in TLC lookup", 0)],
        columns=r.columns)
    r = pd.concat([extra, r], ignore_index=True)
    r.to_csv(f"{fin}/CLEANING_RULES.csv", index=False)
print(r[["rule", "rows_removed"]].to_string(index=False))
print("total removed:", r["rows_removed"].sum())   # expect 9,967,621

removed out_test
                                        rule  rows_removed
                      r01_missing_key_fields             0
          r02_location_id_not_in_zone_lookup             0
r03_malformed_timestamp_outside_study_window           670
                r04_pickup_not_in_file_month          2162
                r05_dropoff_not_after_pickup        607413
                       r06_duration_over_max           998
               r07_zero_or_negative_distance       3470766
                       r08_distance_over_max          7110
                           r09_fare_negative       3836525
                           r10_fare_over_max           880
                r11_negative_charge_or_total         11036
            r12_passenger_count_out_of_range       1948290
                       r13_implausible_speed         81771
total removed: 9967621


In [ ]:
import duckdb
fin = "/content/drive/MyDrive/BigData Project/FINAL_for_partner"
f = f"read_csv_auto('{fin}/agg_daily_borough.csv')"
print(duckdb.sql(f"SELECT pu_borough, sum(trips) AS trips FROM {f} GROUP BY 1 ORDER BY 2 DESC").df())

note = """HANDOFF NOTE - NYC yellow taxi, Jan 2022 to Dec 2025
START HERE: FINAL_for_partner/ (small files). Read DATA_DICTIONARY.csv and CLEANING_RULES.csv first.
Trip-level data: out_20*/cleaned_trips/**/*.parquet (read with DuckDB, hive_partitioning=true).

Counts: 167,858,646 trips in, 157,891,025 kept, 9,967,621 removed (5.9%).

CAVEATS
1. Trips with passenger_count 0 or above 6 (1.2%) and zero-distance trips (2.1%) were removed, so counts run slightly low.
2. Zero-fare trips are KEPT and flagged is_zero_fare (28,238). Filter is_zero_fare = false for fare analysis.
3. Tip % (tip_pct_card) is card-only. Cash tips are not recorded.
4. Weather is NOT in cleaned_trips. Join weather_daily.csv on pickup_date = date. It is already in the two aggregates.
5. congestion_surcharge (older NYS charge, all years) is not cbd_congestion_fee (Manhattan toll, from 5 Jan 2025).
6. share_with_cbd_fee in the aggregates is a share per group. When averaging across days, weight by trips.
7. Zone IDs 264/265 (Unknown/N/A) are kept. Check the borough labels in the aggregates.
8. Weather is from one station (Central Park), so it is the same for all boroughs.
"""
open(f"{fin}/HANDOFF_NOTE.txt", "w").write(note)
print("HANDOFF_NOTE.txt written")

      pu_borough        trips
0      Manhattan  139271104.0
1         Queens   14535585.0
2       Brooklyn    2524036.0
3        Unknown     935728.0
4          Bronx     542271.0
5            N/A      68017.0
6  Staten Island       9102.0
7            EWR       5182.0
HANDOFF_NOTE.txt written


In [ ]:
fin = "/content/drive/MyDrive/BigData Project/FINAL_for_partner"
p = f"{fin}/HANDOFF_NOTE.txt"
s = open(p).read()
old = "7. Zone IDs 264/265 (Unknown/N/A) are kept. Check the borough labels in the aggregates."
new = ("7. Zone IDs 264/265 are kept. In the aggregates they appear as pu_borough = 'Unknown' "
       "(935,728 trips, 0.59%) and 'N/A' (68,017 trips, 0.04%). They are not missing data. "
       "Exclude them for borough-level analysis if needed; the 8 borough labels sum to 157,891,025.")

if old in s:
    open(p, "w").write(s.replace(old, new))
    print("updated")
else:
    print("Handoff note already updated or target text not found")

Handoff note already updated or target text not found


In [ ]:
import pandas as pd
from google.colab import files
src = "/content/drive/MyDrive/BigData Project/FINAL_for_partner/agg_daily_borough.csv"
pd.read_csv(src).to_excel("/content/agg_daily_borough.xlsx", index=False)
files.download("/content/agg_daily_borough.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/BigData Project/FINAL_for_partner/agg_daily_borough.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>